# BiRefNet_lite — Colab T4 benchmark (512 fp32 vs 1024 fp16)

Reproduces the T4 measurement recorded in [`COLAB_T4_REPORT.md`](../COLAB_T4_REPORT.md):
batch 1, one session per pipeline, cold start separated from warm runs, peak
VRAM sampled, masks compared against the 1024 reference.

This notebook is a **reconstruction** written after the 2026-10-26 run, not the
artifact that produced those numbers. It exists to (a) regenerate the report
with machine-captured provenance and (b) close the two gaps that run left open:
**10+ warm runs** and **peak VRAM**.

Runtime → Change runtime type → **T4 GPU** before running.

Outputs land in `smoke-results/benchmark-t4/` in the repo layout.


## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvidia-smi | head -12
import sys; print("python", sys.version.split()[0])

## 2. Install the exact runtime
`onnxruntime-gpu` 1.31.1 to match the recorded run. Colab's preinstalled `onnxruntime` (CPU build) must be replaced.

In [ ]:
!pip install -q --upgrade onnxruntime-gpu==1.31.1 pillow numpy
import onnxruntime as ort
print("onnxruntime", ort.__version__)
print("providers", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), (
    "CUDAExecutionProvider missing — is this a GPU runtime?")

## 3. Point at the repo
Set `REPO` to an existing clone, or clone the URL.

In [ ]:
import os, subprocess, sys

REPO = "/content/remove-photo-bg"
REPO_URL = ""   # e.g. "https://github.com/octoborg-x/remove-photo-bg.git"

if not os.path.isdir(REPO) and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
assert os.path.isdir(REPO), f"clone the repo to {REPO} or set REPO/REPO_URL"
os.chdir(REPO)
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True,
                     text=True).stdout.strip())

## 4. Fetch both models and verify by hash

Content-addressed: each file is checked against the SHA-256 recorded in
`COLAB_T4_REPORT.md` §1. A mismatch aborts rather than benchmarking the wrong
bytes.

In [ ]:
import hashlib, os, urllib.request

MODELS = {
    "512-fp32": {
        "dest": "public/models/model_512.onnx",
        "sha256": "eaafaa10f19c63b77e2bae77bf4b2f59054c208ea08b8c6648f95ef36a3b6b87",
        "bytes": 193514682,
        "urls": [
            "https://huggingface.co/Phoenix-ActuallyFree/BiRefNet_lite_512-ONNX/resolve/main/onnx/model.onnx",
        ],
    },
    "1024-fp16": {
        "dest": "/tmp/hf-model/model_1024_fp16.onnx",
        "sha256": "d39b897ceb16ae654c1731f3dba0cf9b368d9cae74b5a57459b455cc8bfec402",
        "bytes": 114538221,
        "urls": [
            "https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/resolve/"
            "de15b22ba131738a16dff04aab8bdf8dc32e3ac1/onnx/model_fp16.onnx",
            "https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/resolve/"
            "de15b22ba131738a16dff04aab8bdf8dc32e3ac1/model_fp16.onnx",
        ],
    },
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for label, spec in MODELS.items():
    dest = spec["dest"]
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.isfile(dest) and sha256_file(dest) == spec["sha256"]:
        print(f"{label}: already present, hash ok")
        continue
    for url in spec["urls"]:
        try:
            print(f"{label}: downloading {url}")
            urllib.request.urlretrieve(url, dest)
        except Exception as exc:
            print(f"  failed: {exc}")
            continue
        got = sha256_file(dest)
        if got == spec["sha256"]:
            print(f"{label}: ok ({os.path.getsize(dest):,} B)")
            break
        print(f"  hash mismatch: {got} != {spec['sha256']}")
        os.remove(dest)
    else:
        raise SystemExit(
            f"{label}: could not fetch a file matching {spec['sha256']}. "
            f"Upload it to {dest} manually, then re-run this cell.")

print("\nboth models verified")

## 5. Run the benchmark — 10 warm runs, VRAM sampled

`--reps 10` closes the warmth gap; the script now samples
`nvidia-smi memory.used` at 50 ms and writes `peak_vram_mb`, plus a machine
generated `environment` block and each model's `model_sha256`.

In [ ]:
!python3 scripts/benchmark-pipelines.py --providers CUDAExecutionProvider --reps 10 --out smoke-results/benchmark-t4


## 6. Mask quality vs the 1024 reference

In [ ]:
!python3 scripts/compare-models-gpu.py smoke-results/benchmark-t4

## 7. Pull out the numbers for the report

In [ ]:
import json, statistics as st

s = json.load(open("smoke-results/benchmark-t4/summary.json"))
print("ENVIRONMENT")
for k, v in s["environment"].items():
    print(f"  {k}: {v}")

print("\nLATENCY / MEMORY (batch 1)")
for p in s["pipelines"]:
    warm = [w for im in p["images"] for w in im["warm_ms"]]
    vram = [im["peak_vram_mb"] for im in p["images"]]
    print(f"\n  {p['label']}  reps={len(warm)//len(p['images'])}  "
          f"sha256={p['model_sha256']}")
    print(f"    warm mean {st.mean(warm):.1f} ms  median {st.median(warm):.1f}  "
          f"min {min(warm):.1f}  max {max(warm):.1f}  sd {st.stdev(warm):.2f}")
    print(f"    first runs {[im['cold_ms'] for im in p['images']]}")
    print(f"    peak VRAM  {vram}")
    print(f"    peak RSS   {[im['peak_rss_mb'] for im in p['images']]}")

q = json.load(open("smoke-results/benchmark-t4/quality-t4/quality.json"))
print(f"\nQUALITY  mean IoU@0.5 = {q['mean_iou']}   "
      f"mean abs diff = {q['mean_abs_diff']}")
for r in q["rows"]:
    print(f"  {r['image']:22} IoU {r['iou_at_0.5']}")

## 8. Copy the artifacts out

Download these so the numbers can be committed alongside the report.

In [ ]:
import shutil, os
os.makedirs("/content/t4-artifacts", exist_ok=True)
for f in ("summary.json", "quality-t4/quality.json"):
    src = f"smoke-results/benchmark-t4/{f}"
    dst = f"/content/t4-artifacts/{os.path.basename(f)}"
    shutil.copy(src, dst)
    print("copied", dst, os.path.getsize(dst), "B")

# Colab convenience (no-op elsewhere)
try:
    from google.colab import files
    for n in ("summary.json", "quality.json"):
        files.download(f"/content/t4-artifacts/{n}")
except Exception as exc:
    print("not in Colab, skipping download:", exc)